# CS2 EXP-4 — NeoBERT-250M + LoRA


## 1. Dependencies


In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "peft<0.14.0",
    "accelerate",
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
    "einops",   # NeoBERT (chandar-lab/NeoBERT) remote modeling code dependency
], check=True)

# xformers is pinned to match torch==2.5.1 exactly: an unpinned `-U xformers`
# install pulls a newer release that requires a newer torch, silently
# upgrading it out from under this pin. --no-deps stops pip from touching
# torch again to satisfy xformers. flash-attention is not required since
# `use_unpadding=False` in case_study_2/models.py (see PDD sec. 5.2).
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps",
    "xformers==0.0.28.post3",
], check=True)

print("Dependencies installed successfully for CUDA 12.1 driver!")

Dependencies installed successfully for CUDA 12.1 driver!


## 1.5 Settings


In [2]:
import os
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"

CODE_COLUMN = "normalized_code"
#CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

EXP4_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / f"exp4_neobert_lora_v1_{CODE_COLUMN_TAG}"
EXP4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"

HF_TOKEN_VALUE = ""
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
else:
    os.environ.pop("HF_TOKEN", None)

RANK = 16
EPOCHS = 10

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 8

STORAGE_CAP_GB = 60

RUN_SMOKE_TEST = True
RUN_OFFICIAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Downsampled parquet: {DOWNSAMPLED_PARQUET}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
# (Device is detected and printed in the "Verify GPU, RAM, and storage budget"
# cell below -- DEVICE doesn't exist yet at this point in the notebook.)


Settings loaded.
Workspace: /workspace
Repository: /workspace/DiverseVul--IS-Project
Data root: /workspace/IntelligentSystemProject/VulnerabilityDetectionData
Downsampled parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet
Manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_shared_rotating_5fold_v1/project_grouped_5fold_manifest.parquet
Hugging Face cache: /workspace/IntelligentSystemProject/hf_cache


## 2. Clone the repository


In [3]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")

    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")


Repository already exists at /workspace/DiverseVul--IS-Project


## 3. Verify GPU, RAM, and storage budget


In [4]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")

Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 4. Data availability check


In [5]:
required_data_files = {
    "downsampled parquet": DOWNSAMPLED_PARQUET,
    "shared 5-fold manifest": MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by notebooks/scope2_preprocessing.ipynb (downsampling + "
        "manifest-generation sections) and were previously synced through Google Drive. Copy "
        f"them into the paths above, or re-run that notebook. Keep an eye on the {STORAGE_CAP_GB} GB storage cap."
    )
    raise FileNotFoundError("Required processed data/manifest are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


Found downsampled parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet (28.5 MB)
Found shared 5-fold manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_shared_rotating_5fold_v1/project_grouped_5fold_manifest.parquet (0.2 MB)


## 5. Verify required repository files

In [6]:
required_repo_files = [
    SRC_DIR / "case_study_2" / "models.py",
    SRC_DIR / "case_study_2" / "data_loader.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_lora.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_nested_rank.py",
]

missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError("Required EXP-4 files are missing:\n" + "\n".join(missing_repo_files))

print("Required Case Study 2 files are present.")

Required Case Study 2 files are present.


## 6. Import project modules


In [7]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1") or mod_name.startswith("utils"):
        del sys.modules[mod_name]

from case_study_2.models import (
    configure_huggingface_cache,
    load_code_tokenizer,
    DEFAULT_NEOBERT_MODEL,
    DEFAULT_NEOBERT_TOKENIZER,
    count_trainable_parameters,
    CodeSequenceClassifier,
    infer_lora_target_modules,
)
from case_study_2.exp4.exp4_lora import train_lora_model_safe
from case_study_2.exp4.exp4_nested_rank import Exp4Config, run_exp4_nested_rank
from utils import split_manifest
from utils.confidence_intervals import bootstrap_metric_ci, format_ci_report


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 7. Load the downsampled dataset and shared 5-fold manifest

In [8]:
import pandas as pd

if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PATH}")

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(MANIFEST_PATH, config=split_manifest.SplitConfig(n_splits=5, random_state=42))

print("full_df rows:", len(full_df))
print("manifest_df rows:", len(manifest_df))

fold_summary_diag = split_manifest.summarize_manifest(
    manifest_df, config=split_manifest.SplitConfig(n_splits=5, random_state=42)
)
display(fold_summary_diag)

fold_size_ratio = fold_summary_diag["test_rows"].max() / fold_summary_diag["test_rows"].min()
print(f"Fold test-size balance: smallest={fold_summary_diag['test_rows'].min()} rows, "
      f"largest={fold_summary_diag['test_rows'].max()} rows, ratio={fold_size_ratio:.2f}x")
if fold_size_ratio > 2.0:
    print("WARNING: fold sizes are notably imbalanced (ratio > 2x) -- "
          "regenerate the manifest via scope2_preprocessing.ipynb with an updated "
          "DOWNSAMPLE_MAX_ROWS_PER_PROJECT.")
else:
    print("Fold sizes look reasonably balanced.")


full_df rows: 20000
manifest_df rows: 20000


,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,3473,198,3275,0.057011,-0.002039,151,16527,615,0,0.17365,0.197128
1,1,4383,268,4115,0.061145,0.002095,156,15617,610,0,0.21915,0.203655
2,2,4341,224,4117,0.051601,-0.007449,156,15659,610,0,0.21705,0.203655
3,3,4092,253,3839,0.061828,0.002778,151,15908,615,0,0.20460,0.197128
4,4,3711,238,3473,0.064134,0.005084,152,16289,614,0,0.18555,0.198433


Fold test-size balance: smallest=3473 rows, largest=4383 rows, ratio=1.26x
Fold sizes look reasonably balanced.


## 8. Build the dataset frame

In [9]:
required_columns = {"source_row_id", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

dataset_frame = full_indexed.loc[list(manifest_ids)].reset_index(drop=True)
print("dataset_frame rows:", len(dataset_frame))
print("Unique projects:", dataset_frame["project"].nunique())


dataset_frame rows: 20000
Unique projects: 766


## 9. Configuration object


In [10]:
exp4_config = Exp4Config(
    hf_cache_dir=HF_CACHE_DIR,
    code_column=CODE_COLUMN,
    rank=RANK,
    epochs=EPOCHS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    eval_batch_size=EVAL_BATCH_SIZE,
)

print("Code column:", exp4_config.code_column)
print("rank (fixed):", exp4_config.rank)
print("lora_alpha_multiplier:", exp4_config.lora_alpha_multiplier)
print("learning_rate:", exp4_config.learning_rate)
print("inner_n_splits:", exp4_config.inner_n_splits)


Code column: normalized_code
rank (fixed): 16
lora_alpha_multiplier: 2
learning_rate: 0.0002
inner_n_splits: 3


## 10. Sanity-check LoRA target modules


In [11]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_NEOBERT_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

_probe_model = CodeSequenceClassifier(model_name=DEFAULT_NEOBERT_MODEL, freeze_backbone=False, dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred LoRA target_modules:", _target_modules)

del _probe_model
torch.cuda.empty_cache()


2026-09-04 09:59:30.571962: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-04 09:59:30.586289: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788515970.602382    1618 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788515970.607345    1618 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-04 09:59:30.625068: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Inferred LoRA target_modules: ['qkv']


## 11. Smoke test on a small subsample


In [12]:
import time
from sklearn.metrics import average_precision_score

if RUN_SMOKE_TEST:
    sample_df = dataset_frame.sample(n=min(2000, len(dataset_frame)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)

    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")

    t0 = time.time()
    smoke_scores, smoke_model, _ = train_lora_model_safe(
        smoke_train, smoke_val, _tok_check, rank=exp4_config.rank,
        lora_alpha=exp4_config.rank * exp4_config.lora_alpha_multiplier,
        learning_rate=exp4_config.learning_rate, epochs=1,
        batch_size=exp4_config.train_batch_size, grad_accum_steps=exp4_config.grad_accum_steps,
        eval_batch_size=exp4_config.eval_batch_size, num_workers=exp4_config.num_workers,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=exp4_config.code_column,
        max_length=exp4_config.max_length,
    )
    smoke_prauc = float(average_precision_score(smoke_val[exp4_config.label_column].values, smoke_scores))
    print(f"[smoke] PR-AUC={smoke_prauc:.4f} | elapsed={(time.time()-t0)/60:.1f} min")

    del smoke_model
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


[smoke] train=1500 rows | val=500 rows


A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- model.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/workspace/DiverseVul--IS-Project/vuln-detection/src/case_study_2/models.py:72: UserWarning: [models] No known unpadding flag found on the NeoBERT config (checked: ('use_unpadding', 'unpad_inputs', 'unpad', 'pack_sequences')); verify attention-mask correctness manually.
  warnings.warn(


[lora] rank=16 | train_rows=1500 | val_rows=500 | batch_size=32 | grad_accum=2 | steps/epoch=47 | trainable=1,377,025 (0.617%) | total=223,043,330
[lora] epoch 1/1 done | train_loss=1.4471 | val_loss=1.2378 | val_pr_auc=0.1397 <- best so far | epoch_time=0.7 min | total_elapsed=0.7 min | peak_VRAM=0.00 GB
[smoke] PR-AUC=0.1397 | elapsed=1.0 min


## 12. Official rotating 5-fold run

In [13]:
if RUN_OFFICIAL:
    results = run_exp4_nested_rank(
        dataset_frame=dataset_frame,
        manifest=manifest_df,
        config=exp4_config,
        output_dir=EXP4_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "manifest_path": str(MANIFEST_PATH),
        },
    )
    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in results["evaluation"]["pooled_metrics"].items()]))
    print("Mean +/- std across the 5 outer folds (headline result):")
    display(results["evaluation"]["fold_summary"])
    print("Selected hyperparameters by outer fold:")
    display(results["selected"])
else:
    results = None
    print("RUN_OFFICIAL=False; skipping.")


[nested] Starting EXP-4 rotating 5-fold run (NeoBERT LoRA), rank=16 (fixed)
[nested] Threshold calibration: 3-fold inner CV | Refit phase: 10 epoch(s) ceiling, full outer-train

=================== OUTER FOLD 0 (1/5) ===================
[nested] outer_train=16527 rows | outer_val=3473 rows


/workspace/DiverseVul--IS-Project/vuln-detection/src/case_study_2/models.py:72: UserWarning: [models] No known unpadding flag found on the NeoBERT config (checked: ('use_unpadding', 'unpad_inputs', 'unpad', 'pack_sequences')); verify attention-mask correctness manually.
  warnings.warn(


    [lora] rank=16 | train_rows=11317 | val_rows=5210 | batch_size=32 | grad_accum=2 | steps/epoch=354 | trainable=1,377,025 (0.617%) | total=223,043,330
    [lora] VRAM after model load: 0.47 GB
    [lora] epoch 1/10 done | train_loss=1.2098 | val_loss=1.2976 | val_pr_auc=0.1367 <- best so far | epoch_time=4.4 min | total_elapsed=4.5 min | peak_VRAM=27.78 GB
    [lora] epoch 3/10 done | train_loss=1.0427 | val_loss=1.4036 | val_pr_auc=0.1425 <- best so far | epoch_time=4.4 min | total_elapsed=13.3 min | peak_VRAM=27.78 GB
    [lora] epoch 4/10 done | train_loss=0.9628 | val_loss=1.3139 | val_pr_auc=0.1478 <- best so far | epoch_time=4.4 min | total_elapsed=17.8 min | peak_VRAM=27.78 GB
    [lora] epoch 8/10 done | train_loss=0.3584 | val_loss=2.6065 | val_pr_auc=0.1560 | epoch_time=4.3 min | total_elapsed=35.3 min | peak_VRAM=27.78 GB
    [lora] early stopping at epoch 8/10 (no val PR-AUC improvement > 0.0001 for 2 epochs); restoring epoch 6 (val_pr_auc=0.1570).
    [lora] reached max

/workspace/DiverseVul--IS-Project/vuln-detection/src/case_study_2/models.py:72: UserWarning: [models] No known unpadding flag found on the NeoBERT config (checked: ('use_unpadding', 'unpad_inputs', 'unpad', 'pack_sequences')); verify attention-mask correctness manually.
  warnings.warn(


    [lora] rank=16 | train_rows=10570 | val_rows=5957 | batch_size=32 | grad_accum=2 | steps/epoch=331 | trainable=1,377,025 (0.617%) | total=223,043,330
    [lora] VRAM after model load: 0.47 GB
    [lora] epoch 1/10 done | train_loss=1.2340 | val_loss=1.1003 | val_pr_auc=0.1162 <- best so far | epoch_time=6.3 min | total_elapsed=6.5 min | peak_VRAM=27.77 GB
    [lora] epoch 2/10 done | train_loss=1.1432 | val_loss=1.0881 | val_pr_auc=0.1278 <- best so far | epoch_time=4.4 min | total_elapsed=10.8 min | peak_VRAM=27.77 GB
    [lora] epoch 3/10 done | train_loss=1.0812 | val_loss=1.0920 | val_pr_auc=0.1191 | epoch_time=4.2 min | total_elapsed=15.0 min | peak_VRAM=27.77 GB
    [lora] epoch 4/10 done | train_loss=0.9724 | val_loss=1.1460 | val_pr_auc=0.1250 | epoch_time=4.2 min | total_elapsed=19.2 min | peak_VRAM=27.77 GB
    [lora] early stopping at epoch 4/10 (no val PR-AUC improvement > 0.0001 for 2 epochs); restoring epoch 2 (val_pr_auc=0.1278).
    [lora] reached max_epochs=10; res

...

,metric,value
0,n_samples,20000.000000
1,vulnerable_1,1181.000000
2,non_vulnerable_0,18819.000000
3,positive_rate,0.059050
4,threshold,0.622000
5,average_precision_pr_auc,0.128111
6,precision,0.124507
7,recall,0.534293
8,f1,0.201952
9,mcc,0.161778


Mean +/- std across the 5 outer folds (headline result):


,metric,mean,std,min,max
0,positive_rate,0.059144,0.004938,0.051601,0.064134
1,threshold,0.622000,0.000000,0.622000,0.622000
2,average_precision_pr_auc,0.138011,0.007326,0.129466,0.149292
3,precision,0.130512,0.018867,0.112620,0.156194
4,recall,0.544313,0.193271,0.313433,0.723320
5,f1,0.203225,0.009481,0.194483,0.218868
6,mcc,0.168317,0.021366,0.135855,0.188445
7,accuracy,0.748147,0.092819,0.643939,0.844399
8,balanced_accuracy,0.652883,0.043675,0.596206,0.686442
9,specificity,0.761454,0.111443,0.638708,0.878979


Selected hyperparameters by outer fold:


,outer_fold_id,rank,lora_alpha,decision_threshold,inner_validation_f1
0,0,16,32,0.61,0.204826
1,1,16,32,0.63,0.192676
2,2,16,32,0.53,0.204215
3,3,16,32,0.70,0.202779
4,4,16,32,0.64,0.204136


## 13. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [14]:
exp4_oof_ci = bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(format_ci_report(exp4_oof_ci))


average_precision_pr_auc: point estimate = 0.1281
  95% CI (project-block bootstrap): [0.1132, 0.1452]
  valid resamples: 1000/1000 (0 degenerate, dropped)
  n_projects: 766, random_state=42
  Reflects sampling variability within this dataset only; not an estimate of generalization to C functions outside this collection.


## 14. Cleanup

In [15]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP4_OUTPUT_DIR}.")


VRAM allocated: 0.01703936 GB
Disk usage at /workspace: 5129.2 GB used / 5714.2 GB total (296.9 GB free)
